# 05 — Distribution Shift, Experiment 4 (Phase 6)

Objective: freeze scaler/PCA/threshold and measure FPR inflation under (a)
temporal drift, (b) environment shift, (c) cross-dataset transfer on aligned
features. Headline metric: ΔFPR (EDR §4).

In [1]:
import sys
from pathlib import Path

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT))
print("ROOT =", ROOT)

ROOT = D:\Study\Sem 5\AIML lab\pca-network-anomaly-detection-clone


In [2]:
import numpy as np
import pandas as pd

from src import BENIGN_LABEL, CATEGORY_COL, SEED
from src.data_loader import load_dataset
from src.evaluation import evaluate, save_metrics_csv, write_experiment_summary
from src.feature_selection import select_features
from src.pca_detector import PCADetector
from src.preprocessing import Preprocessor, clean_frame, split_one_class
from src.thresholding import fit_threshold, load_thresholds, predict
from src.visualization import plot_shift_overlay

PROC = ROOT / "data" / "processed"
TAB = ROOT / "results" / "tables"
FIG = ROOT / "results" / "figures"
REP = ROOT / "results" / "reports"

# Frozen artefacts from notebooks 02-03: no refit on target, ever (EDR V-02).
pp40 = Preprocessor.load(PROC / "scaler.pkl")
pp38 = Preprocessor.load(PROC / "scaler_common.pkl")
det = PCADetector.load(PROC / "pca_model.pkl")
base = load_thresholds(PROC / "thresholds.json")["p99"]
print(f"frozen M40: k={det.k_} P99={base.value:.2f}")

A = load_dataset(ROOT / "data" / "raw" / "synthetic_A.csv")
B = load_dataset(ROOT / "data" / "raw" / "synthetic_B.csv")
keptA, _ = select_features(A)
Ac, _ = clean_frame(A, keptA)
sp = split_one_class(Ac, seed=SEED)   # identical split: deterministic seed

frozen M40: k=19 P99=23.90


In [3]:
# (a) temporal drift: evening profile applied to the frozen test-benign set.
rng = np.random.default_rng(11)
t1 = sp["test_benign"].copy()
for c, f in (("flow_duration", 2.0), ("flow_bytes_per_s", 1.6),
               ("tot_fwd_pkts", 1.4), ("avg_pkt_size", 1.25),
               ("syn_flag_cnt", 2.0), ("active_mean", 1.5)):
    t1[c] = t1[c] * f * rng.lognormal(0.0, 0.08, len(t1))

s_tr = det.score(pp40.transform(sp["train_benign"]))[0]
s_src = det.score(pp40.transform(sp["test_benign"]))[0]
s_t1 = det.score(pp40.transform(t1))[0]
s_att = det.score(pp40.transform(sp["test_attack"]))[0]
fpr_src = float((s_src > base.value).mean())
yt = np.array([0] * len(s_t1) + [1] * len(s_att))
m4a = evaluate(yt, np.concatenate([s_t1, s_att]), predict(np.concatenate([s_t1, s_att]), base))
print(f"temporal: FPR {fpr_src:.4f} -> {m4a['fpr']:.4f} (d={m4a['fpr']-fpr_src:+.4f}), AUROC={m4a['auroc']:.3f}")
plot_shift_overlay(s_tr, s_t1, s_att, base.value, FIG / "shift_overlay_temporal.png")

temporal: FPR 0.0142 -> 0.0304 (d=+0.0163), AUROC=0.909


WindowsPath('D:/Study/Sem 5/AIML lab/pca-network-anomaly-detection-clone/results/figures/shift_overlay_temporal.png')

In [4]:
# (b)/(c) cross-env model on aligned common features, frozen; tested on B.
det38 = PCADetector().fit(pp38.transform(sp["train_benign"]))
base38 = fit_threshold(det38.score(pp38.transform(sp["train_benign"]))[0], "p99")
Bb = B[B[CATEGORY_COL] == BENIGN_LABEL].reset_index(drop=True)
Ba = B[B[CATEGORY_COL] != BENIGN_LABEL].reset_index(drop=True)
s38_src = det38.score(pp38.transform(sp["test_benign"]))[0]
sb38 = det38.score(pp38.transform(Bb))[0]
sa38 = det38.score(pp38.transform(Ba))[0]
fpr_src38 = float((s38_src > base38.value).mean())
yb = np.array([0] * len(sb38) + [1] * len(sa38))
m4bc = evaluate(yb, np.concatenate([sb38, sa38]), predict(np.concatenate([sb38, sa38]), base38))
print(f"env: FPR {fpr_src38:.4f} -> {m4bc['fpr']:.4f} (d={m4bc['fpr']-fpr_src38:+.4f}), AUROC={m4bc['auroc']:.3f}")
plot_shift_overlay(s38_src, sb38, sa38, base38.value, FIG / "shift_overlay_env.png")

m4 = pd.DataFrame([
    {"shift": "temporal (t0->t1 drift)", "model": "M40", "FPR_source": round(fpr_src, 4),
     "FPR_target": round(m4a["fpr"], 4), "dFPR": round(m4a["fpr"] - fpr_src, 4),
     "recall_tgt": round(m4a["recall"], 4), "AUROC_tgt": round(m4a["auroc"], 4), "AUPRC_tgt": round(m4a["auprc"], 4)},
    {"shift": "environment (A->B)", "model": "M38-common", "FPR_source": round(fpr_src38, 4),
     "FPR_target": round(m4bc["fpr"], 4), "dFPR": round(m4bc["fpr"] - fpr_src38, 4),
     "recall_tgt": round(m4bc["recall"], 4), "AUROC_tgt": round(m4bc["auroc"], 4), "AUPRC_tgt": round(m4bc["auprc"], 4)},
    {"shift": "cross-dataset (A->B, aligned 38)", "model": "M38-common", "FPR_source": round(fpr_src38, 4),
     "FPR_target": round(m4bc["fpr"], 4), "dFPR": round(m4bc["fpr"] - fpr_src38, 4),
     "recall_tgt": round(m4bc["recall"], 4), "AUROC_tgt": round(m4bc["auroc"], 4), "AUPRC_tgt": round(m4bc["auprc"], 4)},
])
save_metrics_csv(TAB / "metrics_exp4.csv", m4)
m4

env: FPR 0.0142 -> 0.0835 (d=+0.0693), AUROC=0.811


,shift,model,FPR_source,FPR_target,dFPR,recall_tgt,AUROC_tgt,AUPRC_tgt
0,temporal (t0->t1 drift),M40,0.0142,0.0304,0.0163,0.6522,0.9089,0.9653
1,environment (A->B),M38-common,0.0142,0.0835,0.0693,0.6525,0.8113,0.8493
2,"cross-dataset (A->B, aligned 38)",M38-common,0.0142,0.0835,0.0693,0.6525,0.8113,0.8493


In [5]:
mode = ("threshold/adaptation problem (AUROC retained)" if m4a["auroc"] > 0.8
        else "representation problem (AUROC collapsed)")
write_experiment_summary(
    REP / "exp4_summary.md", exp_id="Exp4", title="Distribution-shift robustness",
    train_desc="frozen M40 (Exp1) + frozen M38-common", test_desc="drifted t1 benign; env-B benign + B attacks",
    model_desc="frozen PCA, frozen P99 thresholds",
    threshold_desc=f"M40 P99={base.value:.2f}; M38 P99={base38.value:.2f}",
    metrics=m4, figures=["shift_overlay_temporal.png", "shift_overlay_env.png"],
    worked=["attack recall retained while FPR inflates: geometry holds"],
    failed=[f"fixed P99 inflates FPR under shift -> {mode} (EDR §7)"],
    leakage_note="thresholds never fit on target (EDR V-02)",
    notebook="notebooks/05_distribution_shift.ipynb")
print("Exp4 done:", mode)

Exp4 done: threshold/adaptation problem (AUROC retained)


## Checkpoint — Exp4 verdict

- Temporal drift mildly inflates FPR; environment transfer inflates it
  materially while recall/AUROC hold → per the EDR §4 decision rule this is a
  **threshold/adaptation failure, not a representation failure** — the Phase-8
  trigger for adaptive thresholds (README Future Work).